# 4. Sampling

You have a trained denoiser $\varepsilon_\theta$. This notebook uses it to *generate* images: starting from pure Gaussian noise $x_T$ and running the reverse process step by step until we reach $x_0$.

By the end you will have:
- Derived and implemented the DDPM reverse step
- Built a full DDPM sampling loop
- Implemented the faster, deterministic **DDIM sampler**
- Compared quality vs. number of steps for both samplers

In [55]:
# @title Setup — run this cell first (Colab only)
# !git clone https://github.com/maigimenez/let-it-rip
# %cd let-it-rip
# !pip install -q "jax[cuda12]" plotly jaxtyping optax datasets pillow

In [56]:
import sys
import os

# ensure repo root is on the path when opening the notebook from notebooks/
_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
for _p in [os.getcwd(), _root]:
    if _p not in sys.path:
        sys.path.insert(0, _p)

import os
import pickle

import jax
import jax.numpy as jnp
import jax.random as jr
import numpy as np
import optax
import plotly.graph_objects as go
from datasets import load_dataset
from jaxtyping import Array, Float, PRNGKeyArray
from plotly.subplots import make_subplots

print(f"JAX {jax.__version__} · backend: {jax.default_backend()}")

JAX 0.10.1 · backend: cpu


---
## Model recap

The cell below imports the model and schedule from the `solutions` package so this notebook is self-contained. They are given — not exercises.


In [57]:
from solutions.schedule import cosine_schedule, q_sample
from solutions.diffusion import init_params, predict_noise

T = 1000
schedule = cosine_schedule(T)

In [58]:
hf_ds = load_dataset("uoft-cs/cifar10", split="train[:5000]")
images = (
    jnp.array(
        np.stack([np.array(item["img"]) for item in hf_ds]),
        dtype=jnp.float32,
    )
    / 127.5
    - 1.0
)

print(f"images: {images.shape}")

images: (5000, 32, 32, 3)


In [59]:
if os.path.exists("params_nb03.pkl"):
    with open("params_nb03.pkl", "rb") as f:
        params = pickle.load(f)
    print("Loaded checkpoint from notebook 03 ✓")
else:
    print("params_nb03.pkl not found — training a fresh model (5 epochs, quick).")
    print("For better samples, run notebook 03 first and save the checkpoint.")

    optimizer = optax.adam(1e-4)
    params = init_params(jr.PRNGKey(0))
    opt_state = optimizer.init(params)

    def train_step(params, opt_state, x0_batch, key, schedule):
        B = x0_batch.shape[0]
        key_t, key_noise = jr.split(key)
        ts = jr.randint(key_t, (B,), 0, T)
        noise = jr.normal(key_noise, x0_batch.shape)

        def loss_fn(params):
            xt = jax.vmap(q_sample, in_axes=(0, 0, 0, None))(
                x0_batch, ts, noise, schedule["alphas_bar"]
            )
            eps_pred = jax.vmap(predict_noise, in_axes=(None, 0, 0))(params, xt, ts)
            return jnp.mean((noise - eps_pred) ** 2)

        loss, grads = jax.value_and_grad(loss_fn)(params)
        updates, new_opt_state = optimizer.update(grads, opt_state)
        return optax.apply_updates(params, updates), new_opt_state, loss

    train_step_jit = jax.jit(train_step)
    key = jr.PRNGKey(42)
    for epoch in range(1, 6):
        key, sk = jr.split(key)
        perm = jr.permutation(sk, len(images))
        imgs = images[perm]
        for i in range(0, len(imgs) - 128 + 1, 128):
            key, step_key = jr.split(key)
            params, opt_state, loss = train_step_jit(
                params, opt_state, imgs[i : i + 128], step_key, schedule
            )
        print(f"epoch {epoch}/5  loss={float(loss):.4f}")

Loaded checkpoint from notebook 03 ✓


---
## 1. The DDPM sampler

### The reverse step

Given $x_t$, we want to recover $x_{t-1}$. The reverse step uses the posterior $q(x_{t-1} \mid x_t, x_0)$m which is Gaussian with a closed form because *both* $q(x_t \mid x_0)$ and $q(x_{t-1} \mid x_0)$ are Gaussian:

$$q(x_{t-1} \mid x_t, x_0) = \mathcal{N}\!\left(\tilde\mu_t(x_t, x_0),\; \tilde\beta_t\, I\right)$$

with

$$\tilde\mu_t = \frac{\sqrt{\bar\alpha_{t-1}}\,\beta_t}{1-\bar\alpha_t}\,x_0 + \frac{\sqrt{\alpha_t}(1-\bar\alpha_{t-1})}{1-\bar\alpha_t}\,x_t$$

$$\tilde\beta_t = \frac{1-\bar\alpha_{t-1}}{1-\bar\alpha_t}\,\beta_t$$

We don't know $x_0$, but we can substitute our estimate $\hat x_0 = \frac{x_t - \sqrt{1-\bar\alpha_t}\,\varepsilon_\theta(x_t,t)}{\sqrt{\bar\alpha_t}}$. After substitution and simplification the posterior mean becomes:

$$\boxed{\mu_\theta(x_t, t) = \frac{1}{\sqrt{\alpha_t}}\!\left(x_t - \frac{\beta_t}{\sqrt{1-\bar\alpha_t}}\,\varepsilon_\theta(x_t, t)\right)}$$

The full reverse step samples $x_{t-1} \sim \mathcal{N}(\mu_\theta, \tilde\beta_t\,I)$ equivalently:

$$x_{t-1} = \mu_\theta(x_t, t) + \sqrt{\tilde\beta_t}\;z, \qquad z \sim \mathcal{N}(0, I)$$

with **no noise added at the final step** ($t = 0$).

### Exercise 1: implement `ddpm_step`

Given a noisy image `xt`, the predicted noise `eps_pred`, and the current timestep `t`, return $x_{t-1}$.

Steps:
1. Compute the posterior mean $\mu_\theta$ using the boxed formula above
2. Compute the posterior variance $\tilde\beta_t$ — use `jnp.where(t > 0, alphas_bar[t-1], 1.0)` to handle $t = 0$
3. Add $\sqrt{\tilde\beta_t}\,z$ but **zero noise at $t = 0$**

In [60]:
def ddpm_step(
    xt: Float[Array, "h w c"],
    t: int,
    eps_pred: Float[Array, "h w c"],
    schedule: dict,
    key: PRNGKeyArray,
) -> Float[Array, "h w c"]:
    # YOUR CODE HERE
    raise NotImplementedError

In [61]:
# @title 💡 Solution (open to reveal after trying to implement)


def ddpm_step(
    xt: Float[Array, "h w c"],
    t: int,
    eps_pred: Float[Array, "h w c"],
    schedule: dict,
    key: PRNGKeyArray,
) -> Float[Array, "h w c"]:
    betas = schedule["betas"]
    alphas = schedule["alphas"]
    alphas_bar = schedule["alphas_bar"]
    ab_t = alphas_bar[t]

    # Predict x0 and clip to the valid pixel range [-1, 1]. Without this,
    # an imperfect eps_pred gets amplified by the 1/sqrt(alphas[t]) blowup
    # near t = T (cosine schedule pushes alphas[t] close to 0 there), and
    # the chain diverges into pure noise instead of denoising.
    x0_pred = jnp.clip(
        (xt - jnp.sqrt(1.0 - ab_t) * eps_pred) / jnp.sqrt(ab_t), -1.0, 1.0
    )

    # Posterior mean, expressed via the clipped x0 estimate
    ab_prev = jnp.where(t > 0, alphas_bar[t - 1], jnp.array(1.0))
    coef_x0 = jnp.sqrt(ab_prev) * betas[t] / (1.0 - ab_t)
    coef_xt = jnp.sqrt(alphas[t]) * (1.0 - ab_prev) / (1.0 - ab_t)
    mu_t = coef_x0 * x0_pred + coef_xt * xt

    # Posterior variance — treat alphas_bar[-1] as 1 when t = 0
    beta_tilde = (1.0 - ab_prev) / (1.0 - ab_t) * betas[t]

    z = jr.normal(key, xt.shape)
    return mu_t + jnp.where(t > 0, jnp.sqrt(beta_tilde) * z, jnp.zeros_like(xt))

In [62]:
key = jr.PRNGKey(0)
xt_test = jr.normal(key, (32, 32, 3))
eps_zero = jnp.zeros((32, 32, 3))

# At t = 0 the step must be deterministic (no noise term)
out_a = ddpm_step(xt_test, 0, eps_zero, schedule, jr.PRNGKey(1))
out_b = ddpm_step(xt_test, 0, eps_zero, schedule, jr.PRNGKey(99))
assert jnp.allclose(out_a, out_b), "ddpm_step at t=0 should be deterministic"
assert out_a.shape == (32, 32, 3)
print("✓ ddpm_step works")

✓ ddpm_step works


### Exercise 2: implement `ddpm_sample`

Start from $x_T \sim \mathcal{N}(0, I)$ and iterate the reverse step from $t = T-1$ down to $t = 0$.

Fill in the two key lines inside the loop: predict the noise, then call `ddpm_step`.

In [63]:
def ddpm_sample(
    params: dict,
    schedule: dict,
    key: PRNGKeyArray,
    shape: tuple = (32, 32, 3),
) -> Float[Array, "h w c"]:
    key, init_key = jr.split(key)
    xt = jr.normal(init_key, shape)

    for t in reversed(range(T)):
        key, step_key = jr.split(key)
        # YOUR CODE HERE (2 lines)

    return xt

In [64]:
# @title 💡 Solution (open to reveal after trying to implement)


def ddpm_sample(
    params: dict,
    schedule: dict,
    key: PRNGKeyArray,
    shape: tuple = (32, 32, 3),
) -> Float[Array, "h w c"]:
    key, init_key = jr.split(key)
    xt = jr.normal(init_key, shape)

    for t in reversed(range(T)):
        key, step_key = jr.split(key)
        eps_pred = predict_noise(params, xt, t)
        xt = ddpm_step(xt, t, eps_pred, schedule, step_key)

    return xt

In [65]:
# Generate 4 images with DDPM (1000 steps — this takes ~30s on CPU, a few seconds on GPU)
print("Generating with DDPM (1000 steps)…")
key = jr.PRNGKey(5)
ddpm_samples = []
for i in range(4):
    key, sample_key = jr.split(key)
    sample = ddpm_sample(params, schedule, sample_key)
    ddpm_samples.append(np.array(sample))


def to_uint8(arr):
    arr = np.array(arr)
    return np.clip((arr + 1) / 2 * 255, 0, 255).astype(np.uint8)


fig = make_subplots(rows=1, cols=4, horizontal_spacing=0.02)
for i, s in enumerate(ddpm_samples):
    fig.add_trace(go.Image(z=to_uint8(s)), row=1, col=i + 1)
fig.update_xaxes(showticklabels=False).update_yaxes(showticklabels=False)
fig.update_layout(
    title="DDPM samples (1 000 steps) — the MLP denoiser is intentionally weak; DiT in notebook 06 will look much better",
    height=200,
)
fig.show()

Generating with DDPM (1000 steps)…


---
## 2. DDIM (deterministic sampling)

DDPM requires $T = 1000$ reverse steps. Song et al. (2021) showed that you can skip most of them by changing the sampling process **without retraining the denoiser**.

The key insight: the DDPM reverse formula implicitly predicts $\hat x_0$ at each step. DDIM makes this explicit and re-noises directly to any earlier timestep $t' < t$ instead of just $t-1$:

$$\boxed{x_{t'} = \sqrt{\bar\alpha_{t'}}\underbrace{\left(\frac{x_t - \sqrt{1-\bar\alpha_t}\,\hat\varepsilon_\theta}{\sqrt{\bar\alpha_t}}\right)}_{\hat x_0} + \sqrt{1-\bar\alpha_{t'}}\,\hat\varepsilon_\theta}$$

This is fully deterministic (no $z$ term). By choosing a short subsequence of timesteps — e.g. $\{999, 979, \ldots, 19, 0\}$ — we get **50 steps instead of 1000**, often with similar quality.

### Exercise 3: implement `ddim_step`

Given `xt` at timestep `t_from`, the noise prediction, and the target timestep `t_to < t_from`:
1. Compute $\hat x_0$ from `xt` and `eps_pred`
2. Re-noise to `t_to` using the DDIM formula above

In [66]:
def ddim_step(
    xt: Float[Array, "h w c"],
    t_from: int,
    t_to: int,
    eps_pred: Float[Array, "h w c"],
    schedule: dict,
) -> Float[Array, "h w c"]:
    # YOUR CODE HERE
    raise NotImplementedError

In [67]:
# @title 💡 Solution (open to reveal after trying to implement)


def ddim_step(
    xt: Float[Array, "h w c"],
    t_from: int,
    t_to: int,
    eps_pred: Float[Array, "h w c"],
    schedule: dict,
) -> Float[Array, "h w c"]:
    ab_from = schedule["alphas_bar"][t_from]
    ab_to = schedule["alphas_bar"][t_to]

    x_hat0 = (xt - jnp.sqrt(1.0 - ab_from) * eps_pred) / jnp.sqrt(ab_from)
    # Clip to the valid pixel range — otherwise an imperfect eps_pred is
    # amplified by the 1/sqrt(ab_from) blowup near t = T and the chain
    # diverges into pure noise instead of denoising.
    x_hat0 = jnp.clip(x_hat0, -1.0, 1.0)
    return jnp.sqrt(ab_to) * x_hat0 + jnp.sqrt(1.0 - ab_to) * eps_pred

### Exercise 4: implement `ddim_sample`

Build the DDIM sampling loop:
1. Create the timestep subsequence with `np.linspace(T-1, 0, steps+1).round().astype(int)` — this gives `steps+1` evenly-spaced indices from $T-1$ down to $0$
2. Start from $x_T \sim \mathcal{N}(0, I)$
3. For each consecutive pair `(t_from, t_to)` in the sequence: predict noise, call `ddim_step`

*Note*: `ddim_step` is deterministic — no key needed.

In [68]:
def ddim_sample(
    params: dict,
    schedule: dict,
    key: PRNGKeyArray,
    shape: tuple = (32, 32, 3),
    steps: int = 50,
) -> Float[Array, "h w c"]:
    # YOUR CODE HERE
    raise NotImplementedError

In [69]:
# @title 💡 Solution (open to reveal after trying to implement)


def ddim_sample(
    params: dict,
    schedule: dict,
    key: PRNGKeyArray,
    shape: tuple = (32, 32, 3),
    steps: int = 50,
) -> Float[Array, "h w c"]:
    timesteps = np.linspace(T - 1, 0, steps + 1).round().astype(int)

    xt = jr.normal(key, shape)
    for i in range(len(timesteps) - 1):
        t_from = int(timesteps[i])
        t_to = int(timesteps[i + 1])
        eps_pred = predict_noise(params, xt, t_from)
        xt = ddim_step(xt, t_from, t_to, eps_pred, schedule)

    return xt

In [70]:
key = jr.PRNGKey(0)

# DDIM is deterministic — same key always gives the same image
s1 = ddim_sample(params, schedule, key, steps=10)
s2 = ddim_sample(params, schedule, key, steps=10)
assert jnp.allclose(s1, s2), "DDIM must be deterministic"
assert s1.shape == (32, 32, 3)
print("✓ ddim_sample is deterministic and correct shape")

✓ ddim_sample is deterministic and correct shape


---
## 3. Comparing samplers

### Quality vs. number of steps

In [71]:
# Generate 4 images at each step count — all from the same seeds for a fair comparison
seeds = [jr.PRNGKey(i) for i in range(4)]
step_counts = [10, 20, 50, 200]

fig = make_subplots(
    rows=len(step_counts),
    cols=4,
    row_titles=[f"DDIM {s} steps" for s in step_counts],
    horizontal_spacing=0.02,
    vertical_spacing=0.04,
)

for row, steps in enumerate(step_counts, start=1):
    for col, key in enumerate(seeds, start=1):
        img = ddim_sample(params, schedule, key, steps=steps)
        fig.add_trace(go.Image(z=to_uint8(img)), row=row, col=col)

fig.update_xaxes(showticklabels=False).update_yaxes(showticklabels=False)
fig.update_layout(
    title="Same seeds, different step counts — fewer steps = less refinement",
    height=450,
)
fig.show()

### Denoising trajectory

DDIM's determinism lets us inspect the full denoising path. The grid below shows a single sample evolving from pure noise ($x_T$) to the final image ($x_0$).

In [72]:
def ddim_sample_trajectory(
    params, schedule, key, shape=(32, 32, 3), steps=50, n_snapshots=10
):
    """Run DDIM and save intermediate states at log-spaced intervals."""
    timesteps = np.linspace(T - 1, 0, steps + 1).round().astype(int)
    snap_every = max(1, len(timesteps) // n_snapshots)

    xt = jr.normal(key, shape)
    snapshots = [np.array(xt)]

    for i in range(len(timesteps) - 1):
        t_from = int(timesteps[i])
        t_to = int(timesteps[i + 1])
        eps_pred = predict_noise(params, xt, t_from)
        xt = ddim_step(xt, t_from, t_to, eps_pred, schedule)
        if (i + 1) % snap_every == 0:
            snapshots.append(np.array(xt))

    snapshots.append(np.array(xt))
    return snapshots


key = jr.PRNGKey(3)
snaps = ddim_sample_trajectory(params, schedule, key, steps=100, n_snapshots=10)

fig = make_subplots(rows=1, cols=len(snaps), horizontal_spacing=0.01)
labels = ["xₜ (noise)"] + [""] * (len(snaps) - 2) + ["x₀ (final)"]
for i, (snap, label) in enumerate(zip(snaps, labels)):
    fig.add_trace(go.Image(z=to_uint8(snap)), row=1, col=i + 1)
    if label:
        fig.add_annotation(
            x=(i + 0.5) / len(snaps),
            y=-0.1,
            xref="paper",
            yref="paper",
            text=label,
            showarrow=False,
            font=dict(size=11),
        )

fig.update_xaxes(showticklabels=False).update_yaxes(showticklabels=False)
fig.update_layout(
    title="DDIM denoising trajectory (left → right: noise → image)", height=200
)
fig.show()

---
## Bonus. samples from a properly-trained model

The samplers above are correct — the reason the MLP's samples still look noisy is that the *denoiser* (notebook 03's tiny MLP, trained briefly on 5 000 images) is intentionally weak, not that DDPM/DDIM are broken. To see the same sampling math produce real images, this cell loads `google/ddpm-cifar10-32` — a UNet denoiser pretrained on the full CIFAR-10 dataset — and runs it through `diffusers`' own DDPM sampler (same algorithm as `ddpm_sample` above, using a real `ε_θ`).

This checkpoint only ships PyTorch weights, so this one cell uses `torch` + `diffusers` directly instead of the JAX code built in this notebook.

In [73]:
import warnings

warnings.filterwarnings("ignore")

from diffusers import DDPMPipeline
from diffusers.utils import logging as diffusers_logging

diffusers_logging.set_verbosity_error()

pretrained_pipe = DDPMPipeline.from_pretrained("google/ddpm-cifar10-32")
pretrained_out = pretrained_pipe(batch_size=4, num_inference_steps=50)

fig = make_subplots(rows=1, cols=4, horizontal_spacing=0.02)
for i, img in enumerate(pretrained_out.images):
    fig.add_trace(go.Image(z=np.array(img)), row=1, col=i + 1)
fig.update_xaxes(showticklabels=False).update_yaxes(showticklabels=False)
fig.update_layout(
    title="google/ddpm-cifar10-32 — same DDPM algorithm, a properly-trained denoiser",
    height=200,
)
fig.show()

Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


Loading pipeline components...:   0%|          | 0/2 [00:00<?, ?it/s]

  0%|          | 0/50 [00:00<?, ?it/s]

---
## What's next?

Both samplers work with **any** denoiser — you just swap out `predict_noise`. In **05 — Classifier-Free Guidance** you will add a conditioning signal (the class label) so the network can be steered toward a specific category at sampling time:

$$\hat\varepsilon_\text{CFG} = \varepsilon_\theta(x_t, t, \varnothing) + w\bigl(\varepsilon_\theta(x_t, t, c) - \varepsilon_\theta(x_t, t, \varnothing)\bigr)$$

where $c$ is the class condition and $w > 1$ is the guidance scale. Higher $w$ = more faithful to the class, less diversity. The DDIM sampler from this notebook plugs in unchanged.